In [6]:
import numpy as np
import matplotlib.pyplot as plt
import smudgy as sm

In [7]:
N = 100_000
boxsize = 1
dim = 3
pos = np.random.uniform(0, boxsize, (N, dim))
mass = np.ones(N)

pc = sm.PointCloud(pos, mass, boxsize, backend="taichi").\
    global_setup(
        kernel_name="gaussian",
        num_neighbors=8,
        structure="covariant"
        )

[smudgy] Set taichi backend
[smudgy] Using 1 MPI rank
[smudgy] Initialized 3d PointCloud with 100000 particles in periodic box of size=[1 1 1]
[smudgy] Set structure to 'covariant'
[smudgy] Set kernel to 'gaussian'
[smudgy] Set number of neighbors to 8


In [8]:
pc.compute_smoothing()
pc.compute_density()

[smudgy] Building kd-tree from positions
[smudgy] Computing smoothing tensors from 8 neighbors
[smudgy] Computing density using covariant 'gaussian' kernel


In [9]:
vector_field = np.ones((N, dim))
pc.add_fields("vf", vector_field)

for m in ['field', 'gradient']:
    print(m, flush=True)
    f = pc.interpolate(
        pc.weights,
        mode=m
    )
    print(f'done, {f.shape} \n')

for m in ['divergence', 'curl']:
    print(m, flush=True)
    f = pc.interpolate(
        "vf",
        mode=m
    )
    print(f'done, {f.shape} \n')

field
[smudgy] Interpolating fields at query positions using covariant 'gaussian' kernel
done, (100000, 1) 

gradient
[smudgy] Interpolating gradients of fields at query positions using covariant 'gaussian' kernel
done, (100000, 1, 3) 

divergence
[smudgy] Interpolating divergence of fields at query positions using covariant 'gaussian' kernel
done, (100000, 1) 

curl
[smudgy] Interpolating curl of fields at query positions using covariant 'gaussian' kernel
done, (100000, 1, 3) 



In [10]:
f, w = pc.deposit(
    fields="vf",
    averaged=True,
    gridnums=128,
    return_weights=True
)
print(f.shape, w.shape)

1 [128 128 128]


[smudgy] Depositing using covariant 'gaussian' kernel
(3, 128, 128, 128) (128, 128, 128)
